# MTP training on Kaggle (minimal OM-3)

**One-time setup** (notebook sidebar):
- *Session options → Accelerator*: **GPU T4 x1** (not P100, which has no fast fp16). *Internet*: **on**.
- *Add-ons → Secrets*: `GITHUB_TOKEN` (fine-grained token, read-only *Contents* on `rampuriajainam/mtp-indic-wordgroup`) and `HF_TOKEN`. Tick both for this notebook.

**Run overnight:** edit the parameters cell, then *Save Version → Save & Run All (Commit)*. Everything under `/kaggle/working/runs` is kept as the version's output.

**Resume in a new session:** *Add Input →* this notebook's previous version (its output). The restore cell copies every `runs/<run_name>` it finds, and `train.py --resume auto` continues from the latest step.

**Publish a finished run for Om (H10):** create a Dataset from the version output, named `mtp-run-<ID>`.

In [ ]:
# ---- parameters ----
BRANCH = "main"
RUNS = ["R0", "R1", "R2"]          # in order; each resumes if a checkpoint exists
SETS = {}                          # per-run --set overrides, e.g. {"R2": ["head_backbone_grad=0.1"]}
SESSION_BUDGET_MIN = 11 * 60 + 15  # Kaggle kills the session at 12 h; leave time to save outputs

In [ ]:
import base64, glob, json, os, shutil, subprocess, sys, time
from kaggle_secrets import UserSecretsClient

T0 = time.time()
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO = "/tmp/mtp"                  # outside /kaggle/working: the clone is not saved as output
RUN_ROOT = "/kaggle/working/runs"

# Token goes in through env-only git config: never in argv, .git/config or error messages.
auth = base64.b64encode(f"x-access-token:{secrets.get_secret('GITHUB_TOKEN')}".encode()).decode()
git_env = {**os.environ, "GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader",
           "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}", "GIT_TERMINAL_PROMPT": "0"}
shutil.rmtree(REPO, ignore_errors=True)
r = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
                    "https://github.com/rampuriajainam/mtp-indic-wordgroup.git", REPO],
                   env=git_env, capture_output=True, text=True)
del auth, git_env
if r.returncode != 0:
    raise RuntimeError("git clone failed: check the GITHUB_TOKEN secret and BRANCH")
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r /tmp/mtp/requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!cd /tmp/mtp && python -m pytest -q tests 2>&1 | tail -2

In [ ]:
# Restore earlier sessions' runs from any attached input.
os.makedirs(RUN_ROOT, exist_ok=True)
for cfg_path in glob.glob("/kaggle/input/**/runs/*/config.yaml", recursive=True):
    src = os.path.dirname(cfg_path)
    dst = os.path.join(RUN_ROOT, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
        print("restored", dst)

In [ ]:
def run_cmd(cmd, log_path):
    env = {**os.environ, "PYTHONUTF8": "1"}
    with open(log_path, "a", encoding="utf-8") as log, subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env) as p:
        for line in p.stdout:
            log.write(line)
            if "Loading weights" not in line and "it/s]" not in line:
                print(line, end="", flush=True)
    return p.returncode

for run in RUNS:
    left = SESSION_BUDGET_MIN - (time.time() - T0) / 60
    if left < 45:
        print(f"only {left:.0f} min left; skipping {run} (resume it next session)")
        continue
    cmd = [sys.executable, f"{REPO}/scripts/train.py", "--config", f"{REPO}/configs/{run}.yaml",
           "--run_root", RUN_ROOT, "--resume", "auto",
           "--stop_after_min", f"{left - 30:.0f}"]   # ~30 min for data loading, final eval and save
    for s in SETS.get(run, []):
        cmd += ["--set", s]
    print(f"\n>>> {run} ({left:.0f} min left in session)", flush=True)
    rc = run_cmd(cmd, f"/kaggle/working/train_{run}.log")
    print(f">>> {run} exit code {rc}")

In [ ]:
# Last eval per run, and the Gate B check against the laptop numbers (eval_small).
LAPTOP = {"R0": {0: 3.05}, "R1": {0: 3.09, 1: 5.84}}
for path in sorted(glob.glob(f"{RUN_ROOT}/*/metrics.jsonl")):
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    ev = [r for r in rows if r["split"] == "eval" and r["name"].startswith("loss/head")]
    if not ev:
        continue
    step = max(r["step"] for r in ev)
    last = {int(r["name"][9:]): r["value"] for r in ev if r["step"] == step}
    name = os.path.basename(os.path.dirname(path))
    run_id = name.split("_")[0]
    gate = LAPTOP.get(run_id, {})
    print(f"{name} @ step {step}: " + " | ".join(
        f"h{d} {v:.4f}" + (f" (laptop {gate[d]:.2f}, diff {v - gate[d]:+.3f})" if d in gate else "")
        for d, v in sorted(last.items())))